# AdaBoost vs GBDT 对比分析

本 Notebook 从原理、公式推导、代码实现到实验结果，全方位对比 **AdaBoost** 和 **GBDT (Gradient Boosting Decision Tree)**。

## 一、核心区别概览

| 维度 | AdaBoost | GBDT |
|------|----------|------|
| **核心思想** | 自适应调整样本权重，错分样本权重增大 | 每一轮拟合上一轮的**负梯度（残差）** |
| **损失函数** | 指数损失 (Exponential Loss) | 任意可导损失函数（平方损失、对数损失等） |
| **学习方式** | 通过样本权重间接影响模型训练 | 直接对残差/负梯度进行回归拟合 |
| **基学习器** | 通常是浅层决策树（树桩 stump） | 通常是 CART 回归树 |
| **权重更新** | 更新**样本权重** $w_i$ + 计算**分类器权重** $\alpha_t$ | 仅更新**目标值**（残差），不修改样本权重 |
| **组合方式** | 加权投票 $H(x) = \text{sign}(\sum \alpha_t h_t(x))$ | 加法模型 $F(x) = \sum \eta \cdot h_t(x)$ |
| **对异常值** | 敏感（指数损失惩罚大） | 相对鲁棒（可用 Huber 等损失） |
| **过拟合** | 容易过拟合 | 通过学习率、子采样等正则化手段控制 |
| **分类/回归** | 主要用于二分类（有扩展） | 分类、回归、排序均支持 |

## 二、公式推导对比

### 2.1 AdaBoost 算法步骤

**输入**: 训练集 $\{(x_i, y_i)\}_{i=1}^N$, $y_i \in \{-1, +1\}$, 迭代轮数 $T$

1. **初始化样本权重**: $w_i^{(1)} = \frac{1}{N}$

2. **对 $t = 1, 2, \dots, T$**:
   - (a) 在权重分布 $w^{(t)}$ 上训练弱分类器 $h_t(x)$
   - (b) 计算加权错误率: $\epsilon_t = \sum_{i=1}^N w_i^{(t)} \cdot \mathbb{I}(h_t(x_i) \neq y_i)$
   - (c) 计算分类器权重: $\alpha_t = \frac{1}{2} \ln\left(\frac{1 - \epsilon_t}{\epsilon_t}\right)$
   - (d) 更新样本权重: $w_i^{(t+1)} = w_i^{(t)} \cdot \exp(-\alpha_t \cdot y_i \cdot h_t(x_i))$
   - (e) 归一化: $w_i^{(t+1)} = \frac{w_i^{(t+1)}}{\sum_j w_j^{(t+1)}}$

3. **最终强分类器**: $H(x) = \text{sign}\left(\sum_{t=1}^T \alpha_t \cdot h_t(x)\right)$

### 2.2 GBDT 算法步骤

1. **初始化**: $F_0(x) = \arg\min_c \sum_{i=1}^N L(y_i, c)$

2. **对 $t = 1, 2, \dots, T$**:
   - (a) 计算负梯度（伪残差）:
     $$r_{i}^{(t)} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F(x)=F_{t-1}(x)}$$
   - (b) 用 $\{(x_i, r_i^{(t)})\}$ 拟合一棵 CART 回归树，得到叶子节点区域 $R_j^{(t)}, j=1,\dots,J$
   - (c) 对每个叶子节点 $j$ 计算最优输出值:
     $$c_j^{(t)} = \arg\min_c \sum_{x_i \in R_j^{(t)}} L(y_i, F_{t-1}(x_i) + c)$$
   - (d) 更新模型: $F_t(x) = F_{t-1}(x) + \eta \sum_{j=1}^J c_j^{(t)} \cdot \mathbb{I}(x \in R_j^{(t)})$

3. **最终模型**: $F_T(x) = F_0(x) + \sum_{t=1}^T \eta \sum_{j=1}^J c_j^{(t)} \cdot \mathbb{I}(x \in R_j^{(t)})$

> **关键区别**: AdaBoost 修改的是**样本权重分布**，GBDT 修改的是**目标值（拟合残差）**。

## 三、从零实现对比

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import accuracy_score, mean_squared_error

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

### 3.1 手写 AdaBoost (二分类)

In [ ]:
class AdaBoostFromScratch:
    """
    AdaBoost 二分类器
    - 损失函数: 指数损失
    - 基学习器: 决策树桩 (max_depth=1)
    - 核心: 每轮调整样本权重，错分样本权重大 → 下一轮更关注
    """

    def __init__(self, n_estimators=50, learning_rate=1.0):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.alphas = []  # 每个弱分类器的权重
        self.models = []  # 弱分类器列表

    def fit(self, X, y):
        n_samples = X.shape[0]
        w = np.ones(n_samples) / n_samples  # 初始化样本权重

        for t in range(self.n_estimators):
            # (a) 在加权样本上训练弱分类器 (decision stump)
            model = DecisionTreeClassifier(max_depth=1)
            model.fit(X, y, sample_weight=w)
            pred = model.predict(X)

            # (b) 计算加权错误率
            incorrect = (pred != y).astype(float)
            epsilon = np.sum(w * incorrect) / np.sum(w)
            epsilon = max(epsilon, 1e-10)  # 避免除零

            # (c) 计算分类器权重 α
            alpha = self.learning_rate * 0.5 * np.log((1 - epsilon) / epsilon)
            self.alphas.append(alpha)
            self.models.append(model)

            # (d) 更新样本权重: w_i *= exp(-α * y_i * h_t(x_i))
            # y_i ∈ {0,1} 转为 {-1,1}
            y_signed = 2 * y - 1
            pred_signed = 2 * pred - 1
            w = w * np.exp(-alpha * y_signed * pred_signed)
            w = w / np.sum(w)  # 归一化

    def predict(self, X):
        # H(x) = sign(∑ α_t * h_t(x))
        final_pred = np.zeros(X.shape[0])
        for alpha, model in zip(self.alphas, self.models):
            pred = model.predict(X)
            pred_signed = 2 * pred - 1  # {0,1} → {-1,1}
            final_pred += alpha * pred_signed
        return (np.sign(final_pred) + 1) // 2  # {-1,1} → {0,1}

    def predict_proba(self, X):
        final_pred = np.zeros(X.shape[0])
        sum_alpha = sum(self.alphas)
        for alpha, model in zip(self.alphas, self.models):
            pred = model.predict(X)
            pred_signed = 2 * pred - 1
            final_pred += alpha * pred_signed
        prob = 1 / (1 + np.exp(-2 * final_pred / sum_alpha))
        return np.column_stack([1 - prob, prob])

### 3.2 手写 GBDT (回归)

In [ ]:
class GBDTFromScratch:
    """
    GBDT 回归器
    - 损失函数: 平方损失 (MSE)
    - 基学习器: CART 回归树
    - 核心: 每轮拟合上一轮的负梯度（残差），而非修改样本权重
    """

    def __init__(self, n_estimators=50, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.models = []
        self.init_value = None

    def _negative_gradient(self, y, y_pred):
        """
        计算负梯度（伪残差）
        对于平方损失 L = 0.5*(y-F)^2，负梯度 = y - F (即残差)
        """
        return y - y_pred

    def fit(self, X, y):
        # (1) 初始化: F_0(x) = mean(y)
        self.init_value = np.mean(y)
        current_pred = np.full_like(y, self.init_value, dtype=float)

        for t in range(self.n_estimators):
            # (2a) 计算负梯度（残差） ← 关键：不修改样本权重！
            residuals = self._negative_gradient(y, current_pred)

            # (2b) 用回归树拟合残差
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.models.append(tree)

            # (2c) 更新模型: F_t(x) = F_{t-1}(x) + η * h_t(x)
            update = self.learning_rate * tree.predict(X)
            current_pred = current_pred + update

    def predict(self, X):
        y_pred = np.full(X.shape[0], self.init_value, dtype=float)
        for tree in self.models:
            y_pred = y_pred + self.learning_rate * tree.predict(X)
        return y_pred

## 四、实验对比

In [ ]:
# ========== 4.1 分类实验: AdaBoost vs GBDT (sklearn) ==========
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier

X_cls, y_cls = make_classification(
    n_samples=1000, n_features=20, n_informative=10,
    n_redundant=5, random_state=42
)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.3, random_state=42
)

print("分类数据集: 训练集", X_train_c.shape, "测试集", X_test_c.shape)

In [ ]:
# 手写 AdaBoost 分类
ada_scratch = AdaBoostFromScratch(n_estimators=50)
ada_scratch.fit(X_train_c, y_train_c)
y_pred_ada_scratch = ada_scratch.predict(X_test_c)
acc_ada_scratch = accuracy_score(y_test_c, y_pred_ada_scratch)
print(f"手写 AdaBoost 准确率: {acc_ada_scratch:.4f}")

In [ ]:
# sklearn 对比
ada_sk = AdaBoostClassifier(n_estimators=50, algorithm='SAMME', random_state=42)
ada_sk.fit(X_train_c, y_train_c)
acc_ada_sk = accuracy_score(y_test_c, ada_sk.predict(X_test_c))

gbdt_cls = GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=42)
gbdt_cls.fit(X_train_c, y_train_c)
acc_gbdt_cls = accuracy_score(y_test_c, gbdt_cls.predict(X_test_c))

print(f"sklearn AdaBoost 准确率: {acc_ada_sk:.4f}")
print(f"sklearn GBDT 准确率:     {acc_gbdt_cls:.4f}")

In [ ]:
# ========== 4.2 回归实验: GBDT (AdaBoost不擅长回归) ==========
from sklearn.ensemble import GradientBoostingRegressor

X_reg, y_reg = make_regression(n_samples=500, n_features=10, noise=15, random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42
)

# 手写 GBDT 回归
gbdt_scratch = GBDTFromScratch(n_estimators=100, learning_rate=0.1, max_depth=3)
gbdt_scratch.fit(X_train_r, y_train_r)
y_pred_gbdt_scratch = gbdt_scratch.predict(X_test_r)
mse_scratch = mean_squared_error(y_test_r, y_pred_gbdt_scratch)
print(f"手写 GBDT 回归 MSE: {mse_scratch:.4f}")

In [ ]:
# sklearn GBDT 回归对比
gbdt_reg = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gbdt_reg.fit(X_train_r, y_train_r)
mse_sk = mean_squared_error(y_test_r, gbdt_reg.predict(X_test_r))
print(f"sklearn GBDT 回归 MSE: {mse_sk:.4f}")

## 五、可视化对比分析

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ===== (a) 样本权重演化（AdaBoost） =====
ax1 = axes[0, 0]
n_show = 10
weights_over_time = np.zeros((50, n_show))
w = np.ones(len(y_train_c)) / len(y_train_c)
for t in range(50):
    weights_over_time[t] = w[:n_show] * n_show  # 归一化到 [0, n_show]
    model = DecisionTreeClassifier(max_depth=1)
    model.fit(X_train_c, y_train_c, sample_weight=w)
    pred = model.predict(X_train_c)
    incorrect = (pred != y_train_c).astype(float)
    epsilon = np.sum(w * incorrect) / np.sum(w)
    epsilon = max(epsilon, 1e-10)
    alpha = 0.5 * np.log((1 - epsilon) / epsilon)
    y_signed = 2 * y_train_c - 1
    pred_signed = 2 * pred - 1
    w = w * np.exp(-alpha * y_signed * pred_signed)
    w = w / np.sum(w)

for i in range(n_show):
    ax1.plot(range(50), weights_over_time[:, i], alpha=0.7, linewidth=1)
ax1.set_xlabel('迭代轮数')
ax1.set_ylabel('样本权重 (归一化)')
ax1.set_title('AdaBoost: 前10个样本的权重演化\n(错误样本权重上升, 正确样本权重下降)')
ax1.grid(True, alpha=0.3)

# ===== (b) GBDT 残差演化 =====
ax2 = axes[0, 1]
gbdt_reg2 = GradientBoostingRegressor(n_estimators=1, max_depth=3, learning_rate=0.5, random_state=42)
gbdt_reg2.fit(X_train_r, y_train_r)
initial_pred = np.full_like(y_train_r, gbdt_reg2.init_.constant_[0])
residuals_over_time = np.zeros((20, 30))

current = initial_pred.copy()
for t in range(20):
    residuals_over_time[t] = (y_train_r[:30] - current[:30])
    tree = DecisionTreeRegressor(max_depth=3)
    tree.fit(X_train_r, y_train_r[:] - current)
    current = current + 0.5 * tree.predict(X_train_r)

for i in range(30):
    ax2.plot(range(20), residuals_over_time[:, i], alpha=0.5, linewidth=1)
ax2.set_xlabel('迭代轮数')
ax2.set_ylabel('残差值')
ax2.set_title('GBDT: 前30个样本的残差演化\n(残差逐渐收敛到0)')
ax2.grid(True, alpha=0.3)

# ===== (c) AdaBoost 分类器权重 α_t =====
ax3 = axes[0, 2]
ax3.stem(range(len(ada_scratch.alphas)), ada_scratch.alphas, linefmt='C0-', markerfmt='C0o')
ax3.set_xlabel('迭代轮数 t')
ax3.set_ylabel('α_t (分类器权重)')
ax3.set_title('AdaBoost: 每轮分类器权重 α_t\nα_t = 0.5 * ln((1-ε_t)/ε_t)')
ax3.grid(True, alpha=0.3)

# ===== (d) 准确率随迭代轮数变化 =====
ax4 = axes[1, 0]

# AdaBoost accuracy over rounds
ada_acc = []
final_pred_ada = np.zeros(len(y_test_c))
sum_alpha = 0
for alpha, model in zip(ada_scratch.alphas, ada_scratch.models):
    pred = model.predict(X_test_c)
    pred_signed = 2 * pred - 1
    final_pred_ada += alpha * pred_signed
    sum_alpha += alpha
    acc = accuracy_score(y_test_c, (np.sign(final_pred_ada) + 1) // 2)
    ada_acc.append(acc)

ax4.plot(range(1, 51), ada_acc, label='AdaBoost (手写)', linewidth=2)
ax4.set_xlabel('迭代轮数')
ax4.set_ylabel('准确率')
ax4.set_title('分类准确率随迭代变化')
ax4.legend()
ax4.grid(True, alpha=0.3)

# GBDT staged predict for classification
gbdt_cls_acc = []
for y_pred_stage in gbdt_cls.staged_predict(X_test_c):
    gbdt_cls_acc.append(accuracy_score(y_test_c, y_pred_stage))
ax4.plot(range(1, len(gbdt_cls_acc)+1), gbdt_cls_acc, label='GBDT (sklearn)', linewidth=2)
ax4.legend()

# ===== (e) MSE 随迭代轮数变化（回归） =====
ax5 = axes[1, 1]

# GBDT 回归 stage prediction
gbdt_mse = []
for y_pred_stage in gbdt_reg.staged_predict(X_test_r):
    gbdt_mse.append(mean_squared_error(y_test_r, y_pred_stage))

ax5.plot(range(1, len(gbdt_mse)+1), gbdt_mse, linewidth=2)
ax5.set_xlabel('迭代轮数')
ax5.set_ylabel('MSE')
ax5.set_title('GBDT 回归 MSE 随迭代变化\n(误差逐渐下降)')
ax5.grid(True, alpha=0.3)

# ===== (f) AdaBoost 决策边界 (简化2D) =====
ax6 = axes[1, 2]
X_2d, y_2d = make_classification(
    n_samples=300, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, random_state=42
)
ada_2d = AdaBoostClassifier(n_estimators=30, algorithm='SAMME', random_state=42)
ada_2d.fit(X_2d, y_2d)

x_min, x_max = X_2d[:, 0].min()-1, X_2d[:, 0].max()+1
y_min, y_max = X_2d[:, 1].min()-1, X_2d[:, 1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
Z = ada_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

ax6.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
ax6.scatter(X_2d[y_2d==0, 0], X_2d[y_2d==0, 1], c='red', s=15, label='Class 0', edgecolors='k')
ax6.scatter(X_2d[y_2d==1, 0], X_2d[y_2d==1, 1], c='blue', s=15, label='Class 1', edgecolors='k')
ax6.set_title('AdaBoost 决策边界 (2D)')
ax6.legend(fontsize=7)

plt.tight_layout()
plt.savefig('adaboost_vs_gbdt_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存为 adaboost_vs_gbdt_comparison.png")

## 六、总结

### AdaBoost
- **适应性强**: 逐步聚焦难分类样本
- **简单直观**: 仅需调整样本权重
- **局限性**: 对噪声/异常值敏感，不支持任意损失函数

### GBDT
- **灵活通用**: 支持任意可导损失函数（分类/回归/排序）
- **强大鲁棒**: 通过学习率、子采样等正则化防止过拟合
- **广泛应用**: XGBoost、LightGBM、CatBoost 都是基于 GBDT 框架

### 一句话总结
> **AdaBoost 改变样本权重适应数据，GBDT 拟合残差逐步逼近目标。** 前者是"哪些样本更难"的视角，后者是"误差还剩多少"的视角。